# 🎓 MsAdmit: Model Training & 3-3-3 Portfolio Recommender

This notebook demonstrates:
1. Loading cleaned canonical admissions data (58,274 records across 4 platforms).
2. Training & benchmarking **Logistic Regression, Random Forest, LightGBM, and XGBoost**.
3. Probability Calibration with Platt Scaling (`CalibratedClassifierCV`).
4. Vectorized **Portfolio Recommender Engine**: Evaluating an applicant across 273 universities to deliver the **Top 3 Target, Top 3 Consider, and Top 3 Reach** programs.
5. Explainable AI with **SHAP TreeExplainer** and What-If Counterfactual Simulator.

In [ ]:
import os
import sys
import ctypes
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Preload OpenMP on macOS
for lib_candidate in ["/opt/homebrew/opt/libomp/lib/libomp.dylib", "/usr/local/opt/libomp/lib/libomp.dylib"]:
    if os.path.exists(lib_candidate):
        try:
            ctypes.CDLL(lib_candidate)
            break
        except Exception:
            pass

sys.path.append(os.path.abspath("../src"))
from recommender import PortfolioRecommender
from explain import AdmissionExplainer

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

### 1. Model Benchmark Results
We trained 4 classification architectures on 58,274 records with an 80/20 stratified split:

In [ ]:
with open("../models/model_features.json", "r") as f:
    meta = json.load(f)

print("Model Features:", meta["features"])
print("\nCalibrated XGBoost Production Metrics:")
for k, v in meta["metrics"].items():
    print(f"  {k:<15}: {v}")

### 2. Run Portfolio Recommendation (Top 3 Target, Consider, Reach)
Let's test an applicant profile with:
- Undergrad GPA: 3.65 / 4.0
- GRE: 321
- TOEFL: 106
- Work Experience: 1.5 years
- Internships: 1
- Research Experience: 1 year
- Research Publications: 1
- SOP LLM Score: 82 / 100

In [ ]:
recommender = PortfolioRecommender(
    model_path="../models/xgboost_admissions_model.joblib",
    catalog_path="../data/processed/university_catalog.json",
    features_meta_path="../models/model_features.json"
)

applicant_profile = {
    "gpa": 3.65,
    "gre": 321.0,
    "toefl": 106.0,
    "work_experience_years": 1.5,
    "internship_experience_years": 1.0,
    "research_experience_years": 1.0,
    "research_publications": 1,
    "sop_score": 82.0
}

result = recommender.recommend_portfolio(applicant_profile, top_k=3)

for category in ["target", "consider", "reach"]:
    print(f"\n=== {category.upper()} (Top 3) ===")
    for uni in result["portfolio"][category]:
        print(f"  • {uni['display_name']} (QS #{uni['qs_rank']}) - Fit Score: {uni['fit_score']}%")
        if uni.get("top_strengths"):
            print(f"    + Key Strength: {uni['top_strengths'][0]['label']}")
        if uni.get("top_bottlenecks"):
            print(f"    - Key Detractor: {uni['top_bottlenecks'][0]['label']}")

### 3. What-If Counterfactual Simulation (Profile Optimizer)
Let's simulate what happens if the applicant retakes the GRE (+5 points, 321 -> 326) and refines their SOP (+8 points, 82 -> 90):

In [ ]:
what_if_levers = {"gre": 326.0, "sop_score": 90.0}
sim_outcome = recommender.simulate_what_if(applicant_profile, what_if_levers)

print("What-If Levers Tested:", sim_outcome["levers_tested"])
for change in sim_outcome["impact_on_portfolio"]:
    print(f"  • {change['university']}: {change['old_fit']}% -> {change['new_fit']}% ({change['delta']})")